In [2]:

from pathlib import Path
import pandas as pd

# Project directory
PROJECT_DIR = Path.cwd().parent

# File paths
RAW_DATA = PROJECT_DIR / "data" / "raw" / "lpmc.dat"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

# Create output directory if needed
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Raw data exists:", RAW_DATA.exists())


Project directory: /Users/vahedbarzegari/Papers/travel-choice-modeling
Raw data exists: True


In [3]:
# Read original data
df = pd.read_csv(RAW_DATA, sep="\t")

# Basic information
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

# Show first five rows
display(df.head())

# Display column names and data types
display(pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values
}))


Number of rows: 81086
Number of columns: 32


,trip_id,household_id,person_n,trip_n,travel_mode,purpose,fueltype,faretype,bus_scale,survey_year,...,dur_pt_access,dur_pt_rail,dur_pt_bus,dur_pt_int,pt_interchanges,dur_driving,cost_transit,cost_driving_fuel,cost_driving_ccharge,driving_traffic_percent
0,0,0,0,0,4,3,1,1,1.0,1,...,0.134444,0.0,0.016667,0.0,0,0.052222,1.5,0.14,0.0,0.111702
1,1,0,0,1,4,3,1,1,1.0,1,...,0.109444,0.0,0.055556,0.0,0,0.059444,1.5,0.15,0.0,0.112150
2,2,0,0,2,4,3,1,1,1.0,1,...,0.203056,0.0,0.210278,0.0,0,0.236667,1.5,0.79,0.0,0.203052
3,3,0,0,3,4,3,1,1,1.0,1,...,0.205556,0.0,0.258611,0.0,0,0.233333,1.5,0.78,0.0,0.160714
4,4,0,1,2,4,3,1,4,1.0,1,...,0.203056,0.0,0.189444,0.0,0,0.229167,1.5,0.78,0.0,0.130909


,column,dtype
0,trip_id,int64
1,household_id,int64
2,person_n,int64
3,trip_n,int64
4,travel_mode,int64
5,purpose,int64
6,fueltype,int64
7,faretype,int64
8,bus_scale,float64
9,survey_year,int64


In [4]:
# Output file
OUTPUT_FILE = PROCESSED_DIR / "lpmc.csv"

# Export without modifying the original data
df.to_csv(OUTPUT_FILE, index=False)

# Verify saved file
df_check = pd.read_csv(OUTPUT_FILE)

print("Saved to:", OUTPUT_FILE)
print("Rows:", len(df_check))
print("Columns:", len(df_check.columns))

assert df_check.shape == df.shape
print("CSV export verified successfully.")


Saved to: /Users/vahedbarzegari/Papers/travel-choice-modeling/data/processed/lpmc.csv
Rows: 81086
Columns: 32
CSV export verified successfully.


In [5]:
# Dataset overview

print(f"Number of observations: {df.shape[0]:,}")
print(f"Number of variables: {df.shape[1]}")

summary = pd.DataFrame({
    "Data Type": df.dtypes,
    "Missing Values": df.isna().sum(),
    "Missing (%)": df.isna().mean() * 100,
    "Unique Values": df.nunique()
})

display(summary)


Number of observations: 81,086
Number of variables: 32


,Data Type,Missing Values,Missing (%),Unique Values
trip_id,int64,0,0.0,81086
household_id,int64,0,0.0,17616
person_n,int64,0,0.0,10
trip_n,int64,0,0.0,19
travel_mode,int64,0,0.0,4
purpose,int64,0,0.0,5
fueltype,int64,0,0.0,6
faretype,int64,0,0.0,5
bus_scale,float64,0,0.0,3
survey_year,int64,0,0.0,3


In [6]:

# Travel mode distribution

mode_labels = {
    1: "Walk",
    2: "Cycle",
    3: "Public Transport",
    4: "Car"
}

mode_counts = df["travel_mode"].value_counts().sort_index()

mode_summary = pd.DataFrame({
    "Mode": mode_counts.index.map(mode_labels),
    "Trips": mode_counts.values,
    "Share (%)": (
        100 * mode_counts.values / len(df)
    ).round(2)
})

display(mode_summary)


,Mode,Trips,Share (%)
0,Walk,14268,17.60
1,Cycle,2405,2.97
2,Public Transport,28605,35.28
3,Car,35808,44.16


In [7]:

# Identify time and cost columns

time_cost_cols = [
    col for col in df.columns
    if "time" in col.lower()
    or "cost" in col.lower()
]

print("Time and cost variables:")
print(time_cost_cols)

display(df[time_cost_cols].describe().T)


Time and cost variables:
['start_time', 'cost_transit', 'cost_driving_fuel', 'cost_driving_ccharge']


,count,mean,std,min,25%,50%,75%,max
start_time,81086.0,13.834781,4.456804,0.0,10.00,14.00,17.00,23.933333
cost_transit,81086.0,1.563212,1.535197,0.0,0.00,1.50,2.40,13.490000
cost_driving_fuel,81086.0,0.832049,0.822737,0.0,0.28,0.54,1.09,10.090000
cost_driving_ccharge,81086.0,1.071159,3.178036,0.0,0.00,0.00,0.00,10.500000


In [8]:

# Find possible person identifier columns

id_cols = [
    col for col in df.columns
    if "id" in col.lower()
]

print("Possible ID columns:")
print(id_cols)


Possible ID columns:
['trip_id', 'household_id']


In [10]:
df["person_id"] = (
    df["household_id"].astype(str)
    + "_"
    + df["person_n"].astype(str)
)

print("Unique persons:", df["person_id"].nunique())

Unique persons: 31954


In [11]:

# 1. Travel duration summary
duration_cols = [
    col for col in df.columns
    if col.startswith("dur_")
]

print("Travel duration summary:")
display(df[duration_cols].describe().T.round(2))

# 2. Zero values in duration and cost variables
check_cols = duration_cols + [
    "cost_transit",
    "cost_driving_fuel",
    "cost_driving_ccharge"
]

zero_summary = pd.DataFrame({
    "Zero Count": (df[check_cols] == 0).sum(),
    "Zero (%)": (
        (df[check_cols] == 0).mean() * 100
    ).round(2)
})

print("Zero value summary:")
display(zero_summary)

# 3. Person identifiers
df["person_id"] = (
    df["household_id"].astype(str)
    + "_"
    + df["person_n"].astype(str)
)

print("Unique households:", df["household_id"].nunique())
print("Unique persons:", df["person_id"].nunique())
print("Unique trips:", df["trip_id"].nunique())


Travel duration summary:


,count,mean,std,min,25%,50%,75%,max
dur_walking,81086.0,1.13,1.12,0.03,0.35,0.72,1.51,9.28
dur_cycling,81086.0,0.36,0.35,0.01,0.12,0.23,0.48,3.05
dur_pt_access,81086.0,0.16,0.09,0.00,0.09,0.14,0.21,1.19
dur_pt_rail,81086.0,0.09,0.18,0.00,0.00,0.00,0.10,1.47
dur_pt_bus,81086.0,0.17,0.19,0.00,0.03,0.11,0.25,2.15
dur_pt_int,81086.0,0.04,0.08,0.00,0.00,0.00,0.08,0.87
dur_driving,81086.0,0.28,0.25,0.00,0.11,0.19,0.37,2.06


Zero value summary:


,Zero Count,Zero (%)
dur_walking,0,0.00
dur_cycling,0,0.00
dur_pt_access,224,0.28
dur_pt_rail,55678,68.67
dur_pt_bus,14537,17.93
dur_pt_int,56668,69.89
dur_driving,0,0.00
cost_transit,26120,32.21
cost_driving_fuel,3,0.00
cost_driving_ccharge,72814,89.80


Unique households: 17616
Unique persons: 31954
Unique trips: 81086


In [12]:

# Travel time comparison by chosen mode

# Create a temporary analysis dataframe
eda = df.copy()

# Convert travel durations from hours to minutes
eda["walk_min"] = eda["dur_walking"] * 60
eda["cycle_min"] = eda["dur_cycling"] * 60
eda["car_min"] = eda["dur_driving"] * 60

eda["pt_min"] = (
    eda["dur_pt_access"]
    + eda["dur_pt_rail"]
    + eda["dur_pt_bus"]
    + eda["dur_pt_int"]
) * 60

# Summary by actual chosen travel mode
time_cols = [
    "walk_min",
    "cycle_min",
    "pt_min",
    "car_min"
]

time_comparison = (
    eda.groupby("travel_mode")[time_cols]
       .median()
       .round(1)
)

time_comparison.index = time_comparison.index.map({
    1: "Walk",
    2: "Cycle",
    3: "Public Transport",
    4: "Car"
})

display(time_comparison)


,walk_min,cycle_min,pt_min,car_min
travel_mode,,,,
Walk,14.0,5.0,10.2,5.0
Cycle,38.6,13.0,22.8,11.6
Public Transport,72.3,23.8,31.4,19.8
Car,46.3,14.6,25.2,11.3


In [13]:

import numpy as np

mode_names = {
    1: "Walk",
    2: "Cycle",
    3: "Public Transport",
    4: "Car"
}

time_cols = [
    "walk_min",
    "cycle_min",
    "pt_min",
    "car_min"
]

# Find the fastest alternative for each trip
eda["fastest_mode"] = (
    eda[time_cols].to_numpy().argmin(axis=1) + 1
)

# Compare actual choice with fastest alternative
eda["chose_fastest"] = (
    eda["travel_mode"] == eda["fastest_mode"]
)

# Percentage by chosen mode
fastest_summary = (
    eda.groupby("travel_mode")["chose_fastest"]
       .agg(["count", "mean"])
)

fastest_summary["mean"] *= 100
fastest_summary.index = fastest_summary.index.map(mode_names)

fastest_summary.columns = [
    "Trips",
    "Chose Fastest (%)"
]

display(fastest_summary.round(2))

print(
    "Overall percentage choosing fastest:",
    round(eda["chose_fastest"].mean() * 100, 2)
)


,Trips,Chose Fastest (%)
travel_mode,,
Walk,14268,0.08
Cycle,2405,30.52
Public Transport,28605,14.69
Car,35808,82.54


Overall percentage choosing fastest: 42.55


In [14]:

# Driving licence and household car ownership
# by chosen travel mode

license_table = pd.crosstab(
    df["travel_mode"],
    df["driving_license"],
    normalize="index"
) * 100

ownership_table = pd.crosstab(
    df["travel_mode"],
    df["car_ownership"],
    normalize="index"
) * 100

license_table.index = license_table.index.map(mode_names)
ownership_table.index = ownership_table.index.map(mode_names)

print("Driving licence distribution (%):")
display(license_table.round(2))

print("Household car ownership distribution (%):")
display(ownership_table.round(2))

# Check whether some car trips have no licence
print("Car trips with driving_license = 0:",
      ((df["travel_mode"] == 4) &
       (df["driving_license"] == 0)).sum())

print("Car trips with car_ownership = 0:",
      ((df["travel_mode"] == 4) &
       (df["car_ownership"] == 0)).sum())


Driving licence distribution (%):


driving_license,0,1
travel_mode,,
Walk,49.27,50.73
Cycle,30.69,69.31
Public Transport,52.30,47.70
Car,23.24,76.76


Household car ownership distribution (%):


car_ownership,0,1,2
travel_mode,,,
Walk,43.96,38.48,17.56
Cycle,42.29,40.37,17.34
Public Transport,51.12,35.74,13.14
Car,5.01,51.78,43.20


Car trips with driving_license = 0: 8322
Car trips with car_ownership = 0: 1795


In [15]:

# Car trips by driving licence status

car_trips = df[df["travel_mode"] == 4].copy()

car_summary = (
    car_trips.groupby("driving_license")
    .agg(
        trips=("trip_id", "count"),
        median_age=("age", "median"),
        median_distance=("distance", "median"),
        median_driving_time=("dur_driving", "median"),
        mean_car_ownership=("car_ownership", "mean")
    )
)

# Convert hours to minutes
car_summary["median_driving_time"] *= 60

display(car_summary.round(2))

# Age distribution of car users without a licence
no_license_car = car_trips[
    car_trips["driving_license"] == 0
]

print("Age distribution:")
display(no_license_car["age"].describe())

print("Number under 17:",
      (no_license_car["age"] < 17).sum())


,trips,median_age,median_distance,median_driving_time,mean_car_ownership
driving_license,,,,,
0,8322,13.0,2697.5,10.55,1.17
1,27486,46.0,3099.5,11.58,1.45


Age distribution:


count    8322.000000
mean       22.810142
std        21.318503
min         5.000000
25%         8.000000
50%        13.000000
75%        30.000000
max        99.000000
Name: age, dtype: float64

Number under 17: 5050


In [16]:

# Mode choice distribution by driving licence

mode_by_license = pd.crosstab(
    df["driving_license"],
    df["travel_mode"],
    normalize="index"
) * 100

mode_by_license = mode_by_license.rename(
    index={0: "No licence", 1: "Has licence"},
    columns={
        1: "Walk",
        2: "Cycle",
        3: "Public Transport",
        4: "Car"
    }
)

display(mode_by_license.round(2))


travel_mode,Walk,Cycle,Public Transport,Car
driving_license,,,,
No licence,22.64,2.38,48.18,26.80
Has licence,14.47,3.33,27.27,54.93


In [17]:

# Compare mode choice by licence among adults only

adults = df[df["age"] >= 18].copy()

adult_mode_by_license = pd.crosstab(
    adults["driving_license"],
    adults["travel_mode"],
    normalize="index"
) * 100

adult_mode_by_license = adult_mode_by_license.rename(
    index={0: "No licence", 1: "Has licence"},
    columns={
        1: "Walk",
        2: "Cycle",
        3: "Public Transport",
        4: "Car"
    }
)

print("Number of adult trips:", len(adults))
display(adult_mode_by_license.round(2))


Number of adult trips: 69224


travel_mode,Walk,Cycle,Public Transport,Car
driving_license,,,,
No licence,23.54,2.73,57.73,16.00
Has licence,14.47,3.33,27.27,54.93


In [18]:

# Mode choice among licensed adults
# by household car ownership category

licensed_adults = df[
    (df["age"] >= 18) &
    (df["driving_license"] == 1)
].copy()

ownership_mode = pd.crosstab(
    licensed_adults["car_ownership"],
    licensed_adults["travel_mode"],
    normalize="index"
) * 100

ownership_mode = ownership_mode.rename(
    index={
        0: "No car",
        1: "Less than 1 car per adult",
        2: "1+ cars per adult"
    },
    columns={
        1: "Walk",
        2: "Cycle",
        3: "Public Transport",
        4: "Car"
    }
)

display(ownership_mode.round(2))

# Number of trips in each ownership category
print("\nNumber of trips by ownership category:")
display(
    licensed_adults["car_ownership"]
    .value_counts()
    .sort_index()
)


travel_mode,Walk,Cycle,Public Transport,Car
car_ownership,,,,
No car,27.09,7.18,59.09,6.64
Less than 1 car per adult,13.06,3.01,24.81,59.12
1+ cars per adult,10.40,1.94,15.56,72.10



Number of trips by ownership category:


car_ownership
0     8390
1    23805
2    17793
Name: count, dtype: int64

In [19]:

# Compare car and public transport travel times
# among licensed adults

analysis = licensed_adults.copy()

analysis["car_time_min"] = analysis["dur_driving"] * 60

analysis["pt_time_min"] = (
    analysis["dur_pt_access"]
    + analysis["dur_pt_rail"]
    + analysis["dur_pt_bus"]
    + analysis["dur_pt_int"]
) * 60

# Positive value: public transport is slower than car
analysis["pt_minus_car"] = (
    analysis["pt_time_min"] - analysis["car_time_min"]
)

summary = analysis.groupby("car_ownership").agg(
    trips=("trip_id", "count"),
    median_car_time=("car_time_min", "median"),
    median_pt_time=("pt_time_min", "median"),
    median_time_difference=("pt_minus_car", "median")
)

summary.index = summary.index.map({
    0: "No car",
    1: "Less than 1 car per adult",
    2: "1+ cars per adult"
})

display(summary.round(2))


,trips,median_car_time,median_pt_time,median_time_difference
car_ownership,,,,
No car,8390,14.20,23.33,6.54
Less than 1 car per adult,23805,12.35,25.27,9.70
1+ cars per adult,17793,11.67,25.38,10.93


In [20]:

import numpy as np
import pandas as pd

# Group trips by the time advantage of car
analysis["time_gap_group"] = pd.cut(
    analysis["pt_minus_car"],
    bins=[-np.inf, 0, 5, 10, 20, np.inf],
    labels=[
        "PT faster or equal",
        "Car faster by 0-5 min",
        "Car faster by 5-10 min",
        "Car faster by 10-20 min",
        "Car faster by 20+ min"
    ]
)

# Percentage choosing car in each group
car_share = pd.pivot_table(
    analysis,
    index="time_gap_group",
    columns="car_ownership",
    values="travel_mode",
    aggfunc=lambda x: (x == 4).mean() * 100,
    observed=True
)

car_share = car_share.rename(columns={
    0: "No car",
    1: "Less than 1 car/adult",
    2: "1+ cars/adult"
})

print("Car choice share (%) by travel time advantage:")
display(car_share.round(2))

# Sample sizes for interpreting each cell
sample_sizes = pd.crosstab(
    analysis["time_gap_group"],
    analysis["car_ownership"]
)

print("Number of trips in each group:")
display(sample_sizes)


Car choice share (%) by travel time advantage:


car_ownership,No car,Less than 1 car/adult,1+ cars/adult
time_gap_group,,,
PT faster or equal,2.15,12.48,22.22
Car faster by 0-5 min,3.77,39.59,47.89
Car faster by 5-10 min,5.29,58.19,70.22
Car faster by 10-20 min,9.61,71.20,81.50
Car faster by 20+ min,19.15,80.81,91.09


Number of trips in each group:


car_ownership,0,1,2
time_gap_group,,,
PT faster or equal,1163,1899,1035
Car faster by 0-5 min,2204,4385,2840
Car faster by 5-10 min,2346,5960,4265
Car faster by 10-20 min,1967,7382,5735
Car faster by 20+ min,710,4179,3918


### Key Findings and Limitations

1. **Travel time and mode choice:** Car choice becomes more common as its travel-time advantage over public transport increases.

2. **Household car ownership:** Substantial differences in car choice remain across ownership categories, even within similar travel-time difference ranges.

3. **Choice availability:** Driving licence and household car ownership do not fully determine whether car travel is available. Car trips are observed among individuals without driving licences and members of car-free households.

4. **Interpretation limitations:** The findings are descriptive and do not establish causal relationships. Travel costs, personal characteristics, trip purpose, alternative availability, and repeated observations from the same individuals must be considered in subsequent modelling.